# 02 Encoder-Decoder Transformer 全流程走读

## 简介

本笔记本介绍原始 Transformer 论文 ("Attention Is All You Need") 的 Encoder-Decoder 架构。
我们将逐步走读 EncoderLayer 和 DecoderLayer 的每一行代码，观察每个子层的张量形状变化，
并运行完整的 Transformer 前向传播。

## 1. 导入

In [ ]:
import torch
from models.transformer.config import TransformerConfig
from models.transformer.encoder import Encoder, EncoderLayer
from models.transformer.decoder import Decoder, DecoderLayer
from models.transformer.model import Transformer
from models.common.attention import MultiHeadAttention
from models.common.normalization import LayerNorm
from models.common.feedforward import FFN
from models.common.positional_encoding import sinusoidal_pe

# 创建小型演示配置
config = TransformerConfig(
    vocab_size=1000, dim=256, n_heads=8, n_layers=2,
    ff_hidden_dim=1024, max_seq_len=128, dropout=0.1
)
print(f"配置: dim={config.dim}, heads={config.n_heads}, layers={config.n_layers}")

## 2. EncoderLayer 走读

EncoderLayer 的 Post-Norm 架构如下：

```
输入 x (B, S, dim)
  │
  ├─→ Self-Attention(x) → Dropout ─┐
  │                                  ├─→ Add(x, attn_out) → LayerNorm ─┐
  └──────────────────────────────────┘                                    │
                                                                         │
  ┌───────────────────────────────────────────────────────────────────────┘
  │
  ├─→ FFN(x) → Dropout ─┐
  │                      ├─→ Add(x, ffn_out) → LayerNorm → 输出
  └──────────────────────┘
```

In [ ]:
# 实例化单层 EncoderLayer 并逐步走读
encoder_layer = EncoderLayer(config)
x = torch.randn(2, 16, config.dim)  # (batch=2, seq=16, dim=256)

print("=" * 60)
print("EncoderLayer 逐步走读")
print("=" * 60)
print(f"输入 x: {list(x.shape)}")

# 步骤 1: Self-Attention
attn_out = encoder_layer.self_attn(x)
print(f"\n1. Self-Attention 输出: {list(attn_out.shape)}")

# 步骤 2: Dropout
attn_out = encoder_layer.dropout1(attn_out)
print(f"2. Dropout 后: {list(attn_out.shape)} (形状不变)")

# 步骤 3: Residual Add + LayerNorm (残差连接 + 层归一化)
residual1 = x + attn_out
x_norm1 = encoder_layer.norm1(residual1)
print(f"3. 残差相加(x + attn_out): {list(residual1.shape)}")
print(f"4. LayerNorm 后: {list(x_norm1.shape)}")

# 步骤 5: FFN (GELU 激活)
ffn_out = encoder_layer.ffn(x_norm1)
print(f"\n5. FFN 输出: {list(ffn_out.shape)}")
ffn_out = encoder_layer.dropout2(ffn_out)

# 步骤 6: Residual Add + LayerNorm
residual2 = x_norm1 + ffn_out
out = encoder_layer.norm2(residual2)
print(f"6. 残差相加 + LayerNorm 后输出: {list(out.shape)}")

print(f"\n输入形状 == 输出形状: {x.shape == out.shape}")

## 3. DecoderLayer 走读

DecoderLayer 比 EncoderLayer 多一个 Cross-Attention 子层。

```
输入 x (B, S_dec, dim)
  │
  ├─→ Masked Self-Attention(x) → Dropout → Add → LayerNorm ─┐
  │                                                            │
  ├─→ Cross-Attention(Q=x, KV=encoder_output) → Dropout → Add → LayerNorm ─┐
  │                                                                          │
  ├─→ FFN(x) → Dropout → Add → LayerNorm → 输出
```

In [ ]:
# 实例化单层 DecoderLayer 并逐步走读
decoder_layer = DecoderLayer(config)

# 准备数据
decoder_input = torch.randn(2, 8, config.dim)   # (B=2, seq=8, dim=256) - 目标序列
encoder_output = torch.randn(2, 16, config.dim)  # (B=2, seq=16, dim=256) - Encoder 输出

print("=" * 60)
print("DecoderLayer 逐步走读")
print("=" * 60)
print(f"Decoder 输入 x:      {list(decoder_input.shape)}")
print(f"Encoder 输出(上下文): {list(encoder_output.shape)}")

# 步骤 1: Masked Self-Attention (带因果掩码)
masked_attn = decoder_layer.self_attn(decoder_input, use_causal_mask=True)
masked_attn = decoder_layer.dropout1(masked_attn)
x = decoder_layer.norm1(decoder_input + masked_attn)
print(f"\n1. Masked Self-Attn → Residual → LN: {list(x.shape)}")

# 步骤 2: Cross-Attention (Q 来自 decoder, K/V 来自 encoder)
q_cross = decoder_layer.cross_attn.w_q(x)
k_cross = decoder_layer.cross_attn.w_k(encoder_output)
v_cross = decoder_layer.cross_attn.w_v(encoder_output)
print(f"\n2. Cross-Attention Q 投影: {list(q_cross.shape)}")
print(f"   Cross-Attention K 投影 (来自Encoder): {list(k_cross.shape)}")
print(f"   Cross-Attention V 投影 (来自Encoder): {list(v_cross.shape)}")

q_cross = decoder_layer.cross_attn._split_heads(q_cross)
k_cross = decoder_layer.cross_attn._split_heads(k_cross)
v_cross = decoder_layer.cross_attn._split_heads(v_cross)
cross_out = decoder_layer.cross_attn._scaled_dot_product_attention(q_cross, k_cross, v_cross)
cross_out = decoder_layer.cross_attn._merge_heads(cross_out)
cross_out = decoder_layer.cross_attn.w_o(cross_out)
cross_out = decoder_layer.dropout2(cross_out)
x = decoder_layer.norm2(x + cross_out)
print(f"   Cross-Attn 输出: {list(cross_out.shape)}")
print(f"   残差相加+LN 后: {list(x.shape)}")

# 步骤 3: FFN
ffn_out = decoder_layer.ffn(x)
ffn_out = decoder_layer.dropout3(ffn_out)
x = decoder_layer.norm3(x + ffn_out)
print(f"\n3. FFN → Residual → LN 最终输出: {list(x.shape)}")

## 4. 完整 Transformer 前向传播

In [ ]:
# 构建完整的 Transformer
model = Transformer(config)

# 准备输入: src=源语言, tgt=目标语言 (例如英译中)
src_ids = torch.randint(0, config.vocab_size, (2, 32))  # 英文句子
tgt_ids = torch.randint(0, config.vocab_size, (2, 16))  # 中文句子

print("完整 Transformer 前向传播:")
print(f"  src_ids (源序列): {list(src_ids.shape)}")
print(f"  tgt_ids (目标序列): {list(tgt_ids.shape)}")

# 前向传播
logits = model(src_ids, tgt_ids)
print(f"\n  logits (输出): {list(logits.shape)}")
print(f"    解释: (B, tgt_seq_len, vocab_size) — 每个目标位置的概率分布")

# 统计参数量
n_params = sum(p.numel() for p in model.parameters())
print(f"\n  总参数量: {n_params:,}")

## 5. 模型结构打印

In [ ]:
print("=" * 60)
print("Transformer 模型结构")
print("=" * 60)

def print_module_tree(module, prefix="", is_last=True):
    """递归打印模块树"""
    connector = "└── " if is_last else "├── "
    children = list(module.named_children())
    name = module.__class__.__name__
    n_param = sum(p.numel() for p in module.parameters())
    print(f"{prefix}{connector}{name} ({n_param:,} params)")

    ext_prefix = "    " if is_last else "│   "
    for i, (child_name, child_module) in enumerate(children):
        child_is_last = (i == len(children) - 1)
        child_prefix = f"{prefix}{ext_prefix}{'└── ' if child_is_last else '├── '}"
        n_cp = sum(p.numel() for p in child_module.parameters())
        print(f"{child_prefix}{child_name}: {child_module.__class__.__name__} ({n_cp:,} params)")

print_module_tree(model)

print("\n" + "=" * 60)
print("Encoder 内部结构")
print("=" * 60)
print_module_tree(model.encoder)

print("\n" + "=" * 60)
print("单层 EncoderLayer 内部")
print("=" * 60)
print_module_tree(model.encoder.layers[0])

## 6. 位置编码可视化

Transformer 使用正弦位置编码 (Sinusoidal PE) 为每个位置提供顺序信息。

In [ ]:
import math

pe = sinusoidal_pe(seq_len=32, dim=64)  # 32 个位置, 64 维

print("正弦位置编码形状: ", list(pe.shape))
print("\n前 8 个位置的编码值 (每 4 个维度取样):")
print("(值域: [-1, 1], 低频维度变化慢, 高频维度变化快)")
print()

# ASCII 可视化
chars = " ░▒▓█"
for pos in range(8):
    row = f"pos{pos:02d} "
    for d in range(0, 64, 2):  # 每 2 维取样
        v = pe[0, pos, d].item()
        idx = int((v + 1) / 2 * (len(chars) - 1))  # 映射到 [0, 4]
        idx = max(0, min(idx, len(chars) - 1))
        row += chars[idx]
    print(row)

print("\n低频维度在左侧（变化缓慢），高频维度在右侧（变化迅速）")

## 8. 相关文档

本 notebook 对应的详细原理文档：

- [Transformer 原理详解](../../docs/models/transformer.md) — Encoder-Decoder 架构、Post-Norm、Sinusoidal PE 的完整原理
- [Attention 机制详解](../../docs/models/attention.md) — Multi-Head Attention 的数学基础

建议阅读顺序：先运行本 notebook 建立直觉，再阅读文档理解数学细节。

## 9. 练习与思考

1. **Post-Norm vs Pre-Norm**: 将 `EncoderLayer` 改为 Pre-Norm 架构（LayerNorm 在 Attention 之前），观察输出差异
2. **去掉 Cross-Attention**: 将 DecoderLayer 的 Cross-Attention 移除，观察模型退化为 Decoder-only 的效果
3. **位置编码对比**: 用 RoPE 替换 Sinusoidal PE，对比两者的外推能力（测试训练 64 长度、推理 128 长度）

## 7. 关键要点总结

1. **EncoderLayer** = Self-Attention + FFN，每个子层后都有残差连接和 LayerNorm（Post-Norm）
2. **DecoderLayer** = Masked Self-Attention + Cross-Attention + FFN
3. **Cross-Attention** 的 Q 来自 Decoder，K/V 来自 Encoder —— 这是 Encoder 信息注入 Decoder 的唯一通道
4. **Post-Norm** 将 LayerNorm 放在子层之后、残差连接之前
5. **正弦位置编码** 通过 sin/cos 函数为每个位置和维度生成唯一编码，无需训练
6. Transformer 是许多现代 LLM 的基础，后续 LLaMA 等对其做了大量改进